### KPIs: fleet utilization %

In [ ]:
from clickhouse_driver import Client

client = Client(
    host="localhost",
    port=int(9000),
    user="click",
    password="click",
    database="pl",
)

with open("../sql/fleet_ur.sql", "r") as f:
    query = f.read()

df = client.query_dataframe(query)
# df

# debug
#
# df[df.fleet_id != 7].describe()
# df[df.trip_hours < 0].describe()

In [ ]:
# df.hist()

In [ ]:
# add fleet-driver utilization rate
df["ur"] = df.trip_hours / 24

# get fleet utilization rate
df_fleet = df.groupby(by=["fleet_id"]).mean("ur").reset_index()

In [ ]:
# df_fleet

In [ ]:
# built plot
import plotly.express as px

# sort by ur
df_plot = df_fleet.sort_values("ur", ascending=False).copy()

# fleet_id to str to keep only existing fleets on x
df_plot["fleet_id"] = df_plot["fleet_id"].astype(str)

# add ur %
df_plot["ur_percent"] = (df_plot["ur"] * 100).round(1)

fig = px.bar(
    df_plot,
    x="fleet_id",
    y="ur",
    color="fleet_id",
    text="ur_percent",
    title="Fleet Utilization Rate (Daily)",
    labels={
        "fleet_id": "Fleet ID",
        "ur": "Utilization Rate",
    },
)

# txt improvement
fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside",
    hovertemplate="<b>Fleet %{x}</b><br>Utilization: %{text}%",
)

fig.update_layout(
    title_font_size=24,
    yaxis=dict(range=[0, 1]),
    legend_title_text="Fleet",
    legend=dict(font=dict(size=12)),
    template="plotly_white",
)

fig.show()